In [4]:
#
# Cell 0: load libraries
#

import numpy as np
from pathlib import Path
import pickle

In [34]:
#
# Cell 1: set path to input data
#

#directory_path = Path("D:/ECG/X_augmented_1000") # chose the right path beforhand
#directory_path = Path("D:/ECG/X_augmented_800") # chose the right path beforhand
#directory_path = Path("D:/ECG/X_augmented_600") # chose the right path beforhand
#directory_path = Path("D:/ECG/X_augmented_400") # chose the right path beforhand
#directory_path = Path("D:/ECG/X_augmented_200") # chose the right path beforhand
#directory_path = Path("D:/ECG/XY_diff_class_3") # chose the right path beforhand
directory_path = Path("D:/ECG/handwrtng_ECG_combined") # chose the right path beforhand
#loaded_arrays = np.load(directory_path/'X_np_arrays.npz',allow_pickle=True)



In [36]:
#
# Cell 2: load X-data
#


#loaded_arrays = np.load(directory_path/'X_np_arrays.npz',allow_pickle=True)
#loaded_arrays = np.load(directory_path/'X_np_arrays_2.npz',allow_pickle=True)
loaded_arrays = np.load(directory_path/'X_np_arrays_3.npz',allow_pickle=True)

# chose the file to restore:
X_train_in = loaded_arrays['a']
X_val_in = loaded_arrays['b']
X_test_in = loaded_arrays['c']

print("X_train_in shape:", X_train_in.shape)
print("X_val_in:", X_val_in.shape)
print("X_test_in:", X_test_in.shape)

X_train_in shape: (15696, 1000, 12)
X_val_in: (5054, 1000, 12)
X_test_in: (2301, 1000, 12)


In [38]:
#
# Cell 3: modify X-data: chose sequence length and lead 
# Note: if number of leads >1 is chosen the models has to be adapted accordingly
#

#chose lead:
lead = 1
#chose lenght of sequence (1000, 800, 600, 400, 200)
seq_length = 1000

X_train = X_train_in[:, 0:seq_length,lead].copy()
X_val = X_val_in[:, 0:seq_length,lead].copy()
X_test = X_test_in[:,0:seq_length,lead].copy()

print("X_train shape:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train shape: (15696, 1000)
X_val: (5054, 1000)
X_test: (2301, 1000)


In [40]:
#
# Cell 4: load y-data 
#

#with open(directory_path/'Y_data.pkl', 'rb') as f:
#   Y_final_dfs = pickle.load(f) # deserialize using load()
#f.close()  

#with open(directory_path/'Y_data_2.pkl', 'rb') as f:
#   Y_final_dfs = pickle.load(f) # deserialize using load()
#f.close()

with open(directory_path/'Y_data_3.pkl', 'rb') as f:
   Y_final_dfs = pickle.load(f) # deserialize using load()
f.close()

#Y_final_dfs

In [42]:
#
# Cell 4a: for control purpose
#


df_control=Y_final_dfs['train_df']
df_control

152      gunpowder exclaim
8370                    SR
12360            all_other
1981                    SR
74         recognize Boxer
               ...        
9089                    SR
8163                    SR
11727            all_other
6581                    SR
1886             all_other
Length: 15696, dtype: object

In [44]:
#
# Cell 5: converts the content of df, column 'codes_2', into a list 
#


y_train=(Y_final_dfs['train_df']).tolist()
y_val=(Y_final_dfs['val_df']).tolist()
y_test=(Y_final_dfs['test_df']).tolist()



#y_train=list(Y_final_dfs['train_df']['codes'])
#y_val=list(Y_final_dfs['val_df']['codes'])
#y_test=list(Y_final_dfs['test_df']['codes'])


#y_train=list(Y_final_dfs['train_df']['codes_2'])
#y_val=list(Y_final_dfs['val_df']['codes_2'])
#y_test=list(Y_final_dfs['test_df']['codes_2'])

len(y_train)

15696

In [48]:
#
# Cell 6: converts the lists of cell 5 into arrays 
#


from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
print(encoder.classes_)
y_val = encoder.transform(y_val)
print(encoder.classes_)
y_test = encoder.transform(y_test)
print(encoder.classes_)
#y_onehot = to_categorical(y_int, num_classes=len(encoder.classes_))
#y_onehot
#y_train[0:50]

['Bagdad unrequited' 'SR' 'The quick brown fox' 'all_other'
 'childish piecemeal' 'fulfilling Dylks' 'gunpowder exclaim'
 'husbandman Jacques' 'jumps over the lazy' 'microscope dusky'
 'overpower lightsome' 'recognize Boxer' 'smart squid who asks']
['Bagdad unrequited' 'SR' 'The quick brown fox' 'all_other'
 'childish piecemeal' 'fulfilling Dylks' 'gunpowder exclaim'
 'husbandman Jacques' 'jumps over the lazy' 'microscope dusky'
 'overpower lightsome' 'recognize Boxer' 'smart squid who asks']
['Bagdad unrequited' 'SR' 'The quick brown fox' 'all_other'
 'childish piecemeal' 'fulfilling Dylks' 'gunpowder exclaim'
 'husbandman Jacques' 'jumps over the lazy' 'microscope dusky'
 'overpower lightsome' 'recognize Boxer' 'smart squid who asks']


In [50]:
#
# Cell 7: calculates class_weight  of classes 
#

from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
print(classes)
cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = {int(c): float(w) for c, w in zip(classes, cw)}
class_weight # copy this to codee module of used model

[ 0  1  2  3  4  5  6  7  8  9 10 11 12]


{0: 10.319526627218934,
 1: 0.17667319511042073,
 2: 9.73697270471464,
 3: 0.16004568072436579,
 4: 10.319526627218934,
 5: 10.408488063660478,
 6: 10.061538461538461,
 7: 10.319526627218934,
 8: 9.896595208070618,
 9: 9.73697270471464,
 10: 9.896595208070618,
 11: 10.232073011734029,
 12: 9.978385251112524}

In [52]:
#
# Cell 8: final preparation of the input data for model training 
#

# -----------------------------
# 0) Prepare data shapes
# -----------------------------

def ensure_3d(X):
    X = np.asarray(X)
    if X.ndim == 2:          # (N, 1000) -> (N, 1000, 1)
        X = X[..., None]
    return X

# Use your arrays here:
# X_train, y_train (0/1), X_val, y_val (0/1)
X_train = ensure_3d(X_train).astype(np.float32)
X_val   = ensure_3d(X_val).astype(np.float32)
X_test   = ensure_3d(X_test).astype(np.float32)
y_train = np.asarray(y_train).astype(np.int32)
y_val   = np.asarray(y_val).astype(np.int32)
y_test   = np.asarray(y_test).astype(np.int32)

input_shape = X_train.shape[1:]  # (1000, 1)

input_shape

(1000, 1)

In [54]:
#
# Cell 8a: for control purpose 
#


print("X_train shape:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)
print("y_test shape:", y_test.shape)


X_train shape: (15696, 1000, 1)
X_val: (5054, 1000, 1)
X_test: (2301, 1000, 1)
y_train shape: (15696,)
y_val shape: (5054,)
y_test shape: (2301,)


In [56]:
#
# Cell 9: creates a folder for data storage.
#

directory_path_input = Path("D:/ECG/input_for_model") # chose the right path beforhand

# Create the directory
try:
    directory_path.mkdir()
    print(f"Directory '{directory_path_input}' created successfully.")
except FileExistsError:
    print(f"Directory '{directory_path_input}' already exists.")
except PermissionError:
    print(f"Permission denied: Unable to create '{directory_pathinput}'.")
except Exception as e:
    print(f"An error occurred: {e}")

Directory 'D:\ECG\input_for_model' already exists.


In [58]:
#
# Cell 10: store data.
#

np.savez_compressed(directory_path_input/'X_y_input.npz', 
                    a=X_train, b=y_train, 
                    c=X_val, d=y_val, 
                    e=X_test, f=y_test)
